### 1. Recuperación semántica de noticias

Esta etapa tiene como objetivo complementar el filtrado léxico realizado mediante palabras clave, identificando noticias potencialmente relacionadas con la temática de análisis que no hayan presentado coincidencias explícitas.

Para ello, se utiliza un modelo de inferencia del lenguaje natural (NLI), que evalúa la compatibilidad semántica entre el contenido textual de una noticia y una hipótesis asociada a la temática configurada.

Las noticias que superen el umbral establecido se incorporarán al conjunto de candidatas para su posterior evaluación de relevancia.

La hipótesis y el umbral se definirán como parámetros configurables, permitiendo su futura administración mediante una tabla de configuración en PostgreSQL.

In [1]:
import pandas as pd
import psycopg
import torch
import os
from dotenv import load_dotenv
from transformers import AutoModelForSequenceClassification, AutoTokenizer

### Configuración de la conexión experimental

Se establece la conexión con la base de datos del caso
experimental, manteniendo aislados los datos utilizados
para la evaluación del IDA respecto del entorno original
del asistente.

In [2]:
# Configuración de la base de datos experimental.
EXPECTED_DATABASE = "ida_caso_02"
ENV_PATH = "../.env.ida_caso_02"

# Cargar las variables del entorno experimental.
if not load_dotenv(ENV_PATH, override=True):
    raise RuntimeError("No se pudo cargar el entorno experimental.")


In [5]:
def get_database_connection():
    """Crea una conexión con la base de datos PostgreSQL configurada.

    Returns:
        psycopg.Connection: Conexión activa a la base de datos PostgreSQL.
    """
    return psycopg.connect(
        host=os.getenv("DB_HOST"),
        port=os.getenv("DB_PORT"),
        dbname=os.getenv("DB_NAME"),
        user=os.getenv("DB_USER"),
        password=os.getenv("DB_PASSWORD"),
    )

In [6]:
# Verificar la conexión con la base experimental.
with get_database_connection() as connection:
    with connection.cursor() as cursor:
        cursor.execute("SELECT current_database();")
        database = cursor.fetchone()[0]

assert database == EXPECTED_DATABASE, (
    f"Base de datos incorrecta: {database}"
)

print(f"Base experimental activa: {database}")

Base experimental activa: ida_caso_02


In [7]:
# Configuración de la temática y del modelo NLI.
NLI_CONFIG = {
    "topic": "sector_energetico_argentino",
    "hypothesis": "La noticia habla sobre el sector energético argentino.",
    "model_name": "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli",
    "threshold": 0.5,
    "max_length": 512,
}

In [8]:
def load_nli_model(model_name):
    """Carga el tokenizador y el modelo NLI en CPU."""
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name
    )

    model = model.float()
    model.eval()

    return tokenizer, model


tokenizer, model = load_nli_model(
    NLI_CONFIG["model_name"]
)

print("Modelo NLI cargado correctamente.")

e:\Trabajo final UGR\communication-intelligence-assistant\.venv\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Modelo NLI cargado correctamente.


In [9]:
def evaluate_nli(
    text,
    hypothesis,
    tokenizer,
    model,
    max_length=512,
):
    """
    Evalúa la relación de entailment entre una noticia
    y una hipótesis temática.

    Devuelve las puntuaciones de entailment, neutral
    y contradiction.
    """
    inputs = tokenizer(
        text,
        hypothesis,
        return_tensors="pt",
        truncation="only_first",
        max_length=max_length,
    )

    with torch.inference_mode():
        outputs = model(**inputs)

    probabilities = torch.softmax(
        outputs.logits,
        dim=-1,
    )[0]

    scores = {
        label.lower(): probabilities[index].item()
        for index, label in model.config.id2label.items()
    }

    return {
        "entailment_score": scores["entailment"],
        "neutral_score": scores["neutral"],
        "contradiction_score": scores["contradiction"],
    }

In [10]:
def get_news_without_keywords(connection):
    """Obtiene noticias sin coincidencias de palabras clave."""
    query = """
            SELECT
                rn.rss_news_id,
                rn.title,
                rn.description,
                rn.published_at
            FROM rss_news AS rn
            WHERE NOT EXISTS (
                SELECT 1
                FROM rss_news_keywords AS rnk
                WHERE rnk.fk_rss_news_id = rn.rss_news_id
            )
            
        """
    

    return pd.read_sql_query(query, connection)

In [11]:
def build_classification_text(row):
    """Construye el texto de clasificación a partir de una noticia."""
    title = row["title"] or ""
    description = row["description"] or ""

    return f"Título: {title}\nDescripción: {description}"


def recover_news_by_nli(
    news,
    config,
    tokenizer,
    model,
):
    """
    Evalúa noticias sin keywords y marca las que superan
    el umbral de recuperación semántica.
    """
    results = news.copy()

    results["classification_text"] = results.apply(
        build_classification_text,
        axis=1,
    )

    scores = []

    for text in results["classification_text"]:
        result = evaluate_nli(
            text=text,
            hypothesis=config["hypothesis"],
            tokenizer=tokenizer,
            model=model,
            max_length=config["max_length"],
        )

        scores.append(result["entailment_score"])

    results["nli_recovery_score"] = scores

    results["nli_recovered"] = (
        results["nli_recovery_score"] >= config["threshold"]
    )

    return results

In [12]:
# Recuperar noticias que no coincidieron con keywords.
connection = get_database_connection()

try:
    news_without_keywords = get_news_without_keywords(
        connection
    )
finally:
    connection.close()

print(
    "Noticias sin keywords:",
    len(news_without_keywords),
)

# Ejecutar la recuperación semántica.
recovery_results = recover_news_by_nli(
    news=news_without_keywords,
    config=NLI_CONFIG,
    tokenizer=tokenizer,
    model=model,
)

# Identificar noticias recuperadas.
recovered_news = recovery_results[
    recovery_results["nli_recovered"]
].copy()

print(
    "Noticias recuperadas mediante NLI:",
    len(recovered_news),
)

display(
    recovered_news[
        [
            "rss_news_id",
            "title",
            "nli_recovery_score",
            "nli_recovered",
        ]
    ].sort_values(
        "nli_recovery_score",
        ascending=False,
    ).round(4)
)

C:\Users\enzor\AppData\Local\Temp\ipykernel_23492\3473948133.py:19: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(query, connection)


Noticias sin keywords: 25
Noticias recuperadas mediante NLI: 0


,rss_news_id,title,nli_recovery_score,nli_recovered


In [13]:
def get_news_with_keywords(connection):
    """Obtiene noticias con al menos una coincidencia de keywords."""
    query = """
        SELECT DISTINCT
            n.rss_news_id,
            n.title,
            n.description
        FROM rss_news AS n
        INNER JOIN rss_news_keywords AS nk
            ON n.rss_news_id = nk.fk_rss_news_id
        ORDER BY n.rss_news_id;
    """

    return pd.read_sql_query(query, connection)




In [14]:
# Recuperar noticias con coincidencias de keywords.
connection = get_database_connection()

try:
    news_with_keywords = get_news_with_keywords(connection)
finally:
    connection.close()

# Unificar las noticias candidatas.
candidate_news = pd.concat(
    [
        news_with_keywords,
        recovered_news[
            ["rss_news_id", "title", "description"]
        ],
    ],
    ignore_index=True,
).drop_duplicates(subset=["rss_news_id"])

print(f"Noticias con keywords: {len(news_with_keywords)}")
print(f"Noticias recuperadas por NLI: {len(recovered_news)}")
print(f"Total de noticias candidatas: {len(candidate_news)}")

Noticias con keywords: 25
Noticias recuperadas por NLI: 0
Total de noticias candidatas: 25


C:\Users\enzor\AppData\Local\Temp\ipykernel_23492\1992892659.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(query, connection)



## 2. Construcción del conjunto de noticias candidatas

Una vez finalizada la recuperación semántica, se construye el conjunto de noticias candidatas que avanzarán a la evaluación de relevancia temática.

Este conjunto se obtiene mediante la unión de dos grupos:

- **Noticias identificadas por palabras clave:** registros que presentan al menos una coincidencia con las keywords configuradas.
- **Noticias recuperadas mediante NLI:** registros que no presentaron coincidencias léxicas, pero superaron el umbral de recuperación semántica.

La combinación utiliza una operación de unión lógica (OR), evitando registros duplicados.

Este procedimiento permite complementar la detección léxica con un mecanismo semántico sin exigir que ambos métodos coincidan.

Las noticias que no superan ninguno de los mecanismos permanecen almacenadas en PostgreSQL, pero no avanzan a la siguiente etapa de procesamiento.


In [15]:

def get_news_with_keywords(connection):
    """
    Recupera las noticias que presentan al menos una
    coincidencia con las palabras clave configuradas.
    """
    query = """
        SELECT DISTINCT
            n.rss_news_id,
            n.title,
            n.description
        FROM rss_news AS n
        INNER JOIN rss_news_keywords AS nk
            ON n.rss_news_id = nk.fk_rss_news_id
        ORDER BY n.rss_news_id;
    """

    return pd.read_sql_query(query, connection)


# Recuperar noticias identificadas mediante keywords.
connection = get_database_connection()

try:
    news_with_keywords = get_news_with_keywords(connection)
finally:
    connection.close()


# Unificar noticias identificadas por keywords y NLI.
candidate_news = pd.concat(
    [
        news_with_keywords,
        recovered_news[
            ["rss_news_id", "title", "description"]
        ],
    ],
    ignore_index=True,
).drop_duplicates(subset=["rss_news_id"])


# Mostrar resumen del conjunto de noticias candidatas.
print(f"Noticias con keywords: {len(news_with_keywords)}")
print(f"Noticias recuperadas por NLI: {len(recovered_news)}")
print(f"Total de noticias candidatas: {len(candidate_news)}")


Noticias con keywords: 25
Noticias recuperadas por NLI: 0
Total de noticias candidatas: 25


C:\Users\enzor\AppData\Local\Temp\ipykernel_23492\1258191492.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(query, connection)



## 3. Evaluación de relevancia temática

### 3.1. Objetivo

La segunda etapa del procesamiento semántico tiene como objetivo asignar un indicador continuo de afinidad temática a las noticias candidatas.

A diferencia de la etapa de recuperación, que utiliza un umbral para determinar si una noticia debe incorporarse al conjunto de candidatas, esta etapa conserva la puntuación obtenida por el modelo NLI.

El indicador permite ordenar y priorizar las noticias según su compatibilidad semántica con la temática configurada, proporcionando información adicional para las etapas posteriores de análisis.

### 3.2. Metodología

Se utiliza el modelo multilingüe `MoritzLaurer/mDeBERTa-v3-base-mnli-xnli`, previamente evaluado durante la fase experimental.

El procesamiento consiste en:

1. Construir el texto de entrada mediante el título y la descripción de cada noticia.
2. Evaluar el texto respecto de la hipótesis temática configurada.
3. Extraer la puntuación correspondiente a la clase *entailment*.
4. Registrar el resultado en el atributo `relevance_score`.

La hipótesis utilizada es:

> La noticia habla sobre inteligencia artificial.

Esta formulación fue seleccionada a partir de los resultados de la experimentación previa, donde presentó un comportamiento más favorable que otras hipótesis evaluadas.

### 3.3. Interpretación del indicador

El atributo `relevance_score` toma valores entre 0 y 1 y representa el grado de compatibilidad que el modelo asigna entre el contenido de la noticia y la hipótesis temática.

Los valores más elevados indican una mayor compatibilidad semántica estimada.

Es importante señalar que esta puntuación no constituye una probabilidad calibrada de relevancia ni una medida de importancia organizacional.

En esta implementación no se establecen categorías discretas ni umbrales adicionales de relevancia. Se conserva la puntuación continua para permitir su análisis y utilización en etapas posteriores.

La configuración de la hipótesis permanece parametrizada para facilitar su futura administración mediante PostgreSQL.


In [16]:

# Configuración de la evaluación de relevancia temática.
RELEVANCE_CONFIG = {
    "topic": NLI_CONFIG["topic"],
    "hypothesis": NLI_CONFIG["hypothesis"],
    "max_length": NLI_CONFIG["max_length"],
}


In [17]:

def evaluate_news_relevance(
    news,
    config,
    tokenizer,
    model,
):
    """
    Evalúa la afinidad semántica de las noticias candidatas
    respecto de la hipótesis temática configurada.

    Parameters
    ----------
    news : pandas.DataFrame
        Noticias candidatas para la evaluación.
    config : dict
        Configuración de la temática y de la hipótesis NLI.
    tokenizer : transformers.PreTrainedTokenizer
        Tokenizador del modelo NLI.
    model : transformers.PreTrainedModel
        Modelo de clasificación NLI.

    Returns
    -------
    pandas.DataFrame
        Noticias con su puntuación de afinidad temática.
    """
    results = news.copy()

    # Construir el texto de entrada para el modelo NLI.
    results["classification_text"] = results.apply(
        build_classification_text,
        axis=1,
    )

    scores = []

    # Evaluar cada noticia respecto de la hipótesis temática.
    for text in results["classification_text"]:
        nli_result = evaluate_nli(
            text=text,
            hypothesis=config["hypothesis"],
            tokenizer=tokenizer,
            model=model,
            max_length=config["max_length"],
        )

        scores.append(nli_result["entailment_score"])

    # Registrar la puntuación continua de afinidad temática.
    results["relevance_score"] = scores

    return results



### 3.4. Ejecución de la evaluación

Se aplica el modelo NLI sobre el conjunto completo de noticias candidatas, independientemente del mecanismo mediante el cual fueron identificadas.

Cada noticia recibe una puntuación de afinidad temática, que se incorpora al conjunto de resultados para su posterior consulta y almacenamiento.

Esta etapa no descarta noticias ni modifica su clasificación de entrada.


In [18]:

# Evaluar la afinidad temática de las noticias candidatas.
relevance_results = evaluate_news_relevance(
    news=candidate_news,
    config=RELEVANCE_CONFIG,
    tokenizer=tokenizer,
    model=model,
)

print(
    f"Noticias evaluadas: {len(relevance_results)}"
)


Noticias evaluadas: 25



### 3.5. Inspección de resultados

Se ordenan las noticias según su puntuación de afinidad temática para inspeccionar el comportamiento del modelo.

Adicionalmente, se calculan estadísticas descriptivas de las puntuaciones obtenidas, incluyendo media, desviación estándar, mediana y percentiles.

Esta inspección permite identificar la distribución de los valores generados y detectar posibles comportamientos atípicos.

Los resultados de esta sección tienen carácter descriptivo. La evaluación formal del rendimiento del modelo se desarrolla de manera independiente en el notebook de experimentación.


In [19]:

# Ordenar noticias por puntuación de afinidad temática.
relevance_ranking = relevance_results.sort_values(
    by="relevance_score",
    ascending=False,
)

# Visualizar las noticias ordenadas por relevancia.
display(
    relevance_ranking[
        [
            "rss_news_id",
            "title",
            "relevance_score",
        ]
    ].round(4)
)

# Obtener estadísticas descriptivas del indicador.
display(
    relevance_results["relevance_score"].describe().round(4)
)


,rss_news_id,title,relevance_score
0,1,Energía Costa Serena: amplía la generación solar,0.9979
6,7,Energía Costa Serena: publica el programa de m...,0.9966
20,24,Red Federal del Sur: informa daños por una tor...,0.8597
15,16,Red Federal del Sur: suspende un proyecto eólico,0.7012
4,5,Energía Costa Serena: demora obras de transpor...,0.4942
19,23,Red Federal del Sur: detecta una fuga de gas,0.2778
13,14,Energía Costa Serena: reporta una avería en un...,0.1258
5,6,Energía Costa Serena: registra pérdidas de com...,0.1037
3,4,Energía Costa Serena: sufre una falla en una s...,0.0933
1,2,Energía Costa Serena: reduce interrupciones el...,0.0756


count    25.0000
mean      0.1941
std       0.3318
min       0.0004
25%       0.0044
50%       0.0179
75%       0.1258
max       0.9979
Name: relevance_score, dtype: float64


## 4. Resultados del procesamiento semántico

El procesamiento NLI genera dos conjuntos de resultados complementarios:

- **`recovery_results`:** contiene las noticias que no presentaron coincidencias con keywords, junto con sus puntuaciones NLI y el indicador booleano de recuperación semántica.
- **`relevance_results`:** contiene las noticias candidatas identificadas mediante keywords o recuperación NLI, junto con su puntuación continua de afinidad temática.

Ambos conjuntos permiten conservar la trazabilidad de las decisiones tomadas durante el procesamiento.

La separación entre recuperación y evaluación de relevancia facilita el análisis de cada etapa, así como la futura incorporación de nuevas temáticas, hipótesis y configuraciones.

Como siguiente actividad, se contempla la persistencia de los resultados en PostgreSQL mediante una estructura independiente de las tablas utilizadas para almacenar noticias y coincidencias léxicas.

Esta estructura permitirá consultar los resultados semánticos sin alterar los datos originales y facilitará su integración con los componentes posteriores del sistema.



## 5. Consolidación de resultados semánticos

Las etapas de recuperación y evaluación de relevancia generan conjuntos de resultados complementarios.

La etapa de recuperación procesa las noticias que no presentaron coincidencias con palabras clave, mientras que la evaluación de relevancia procesa las noticias candidatas identificadas mediante keywords o recuperación semántica.

Dado que ambas etapas utilizan actualmente el mismo modelo, hipótesis y representación textual, sus puntuaciones NLI son comparables y pueden consolidarse en un único conjunto de resultados.

La consolidación permite almacenar una única puntuación por noticia y temática, junto con los indicadores de coincidencia léxica y recuperación semántica.

Las noticias no seleccionadas como candidatas también se conservan para facilitar la trazabilidad del procesamiento y futuros análisis.


In [20]:
# Recuperar los resultados de la etapa 2 correspondientes
# a noticias identificadas mediante keywords.
keyword_results = relevance_results[
    relevance_results["rss_news_id"].isin(
        news_with_keywords["rss_news_id"]
    )
][["rss_news_id", "relevance_score"]].copy()

keyword_results = keyword_results.rename(
    columns={"relevance_score": "nli_score"}
)

keyword_results["keyword_match"] = True
keyword_results["nli_recovered"] = False

# Recuperar los resultados de la etapa 1.
semantic_results = recovery_results[
    ["rss_news_id", "nli_recovery_score", "nli_recovered"]
].copy()

semantic_results = semantic_results.rename(
    columns={"nli_recovery_score": "nli_score"}
)

semantic_results["keyword_match"] = False

# Consolidar ambos conjuntos sin duplicar noticias.
nli_news_results = pd.concat(
    [keyword_results, semantic_results],
    ignore_index=True,
)

# Incorporar la configuración utilizada.
nli_news_results["topic"] = NLI_CONFIG["topic"]
nli_news_results["hypothesis"] = NLI_CONFIG["hypothesis"]
nli_news_results["model_name"] = NLI_CONFIG["model_name"]
nli_news_results["recovery_threshold"] = NLI_CONFIG["threshold"]

# Comprobar que cada noticia aparece una sola vez.
assert not nli_news_results["rss_news_id"].duplicated().any(), (
    "Se encontraron noticias duplicadas."
)

print(f"Noticias consolidadas: {len(nli_news_results)}")
print(f"Con keywords: {nli_news_results['keyword_match'].sum()}")
print(f"Recuperadas por NLI: {nli_news_results['nli_recovered'].sum()}")

Noticias consolidadas: 50
Con keywords: 25
Recuperadas por NLI: 0



## 6. Persistencia de resultados en PostgreSQL

Los resultados consolidados se almacenan en la tabla `nli_news`, manteniendo una relación con las noticias originales de `rss_news`.

Para permitir ejecuciones repetidas del notebook sin generar duplicados, se utiliza una operación de inserción o actualización (*upsert*) basada en la combinación de identificador de noticia y temática.

Cada ejecución actualiza la puntuación NLI, los indicadores de detección y los parámetros utilizados.

Esta estrategia facilita la actualización de resultados cuando cambian las palabras clave, la hipótesis, el modelo o el umbral de recuperación.


In [21]:
def save_nli_results(connection, results):
    """
    Inserta o actualiza los resultados del procesamiento NLI
    en PostgreSQL.

    Parameters
    ----------
    connection : psycopg.Connection
        Conexión activa a PostgreSQL.
    results : pandas.DataFrame
        Resultados consolidados del procesamiento semántico.
    """
    query = """
        INSERT INTO nli_news (
            rss_news_id,
            topic,
            hypothesis,
            model_name,
            nli_score,
            keyword_match,
            nli_recovered,
            recovery_threshold,
            evaluated_at
        )
        VALUES (
            %(rss_news_id)s,
            %(topic)s,
            %(hypothesis)s,
            %(model_name)s,
            %(nli_score)s,
            %(keyword_match)s,
            %(nli_recovered)s,
            %(recovery_threshold)s,
            NOW()
        )
        ON CONFLICT (rss_news_id, topic)
        DO UPDATE SET
            hypothesis = EXCLUDED.hypothesis,
            model_name = EXCLUDED.model_name,
            nli_score = EXCLUDED.nli_score,
            keyword_match = EXCLUDED.keyword_match,
            nli_recovered = EXCLUDED.nli_recovered,
            recovery_threshold = EXCLUDED.recovery_threshold,
            evaluated_at = NOW();
    """

    records = results[
        [
            "rss_news_id",
            "topic",
            "hypothesis",
            "model_name",
            "nli_score",
            "keyword_match",
            "nli_recovered",
            "recovery_threshold",
        ]
    ].to_dict(orient="records")

    # Convertir los tipos de pandas a tipos nativos de Python.
    for record in records:
        record["rss_news_id"] = int(record["rss_news_id"])
        record["nli_score"] = float(record["nli_score"])
        record["keyword_match"] = bool(record["keyword_match"])
        record["nli_recovered"] = bool(record["nli_recovered"])
        record["recovery_threshold"] = float(
            record["recovery_threshold"]
        )

    with connection.cursor() as cursor:
        cursor.executemany(query, records)

    connection.commit()

In [22]:
# Guardar los resultados consolidados en PostgreSQL.
connection = get_database_connection()

try:
    save_nli_results(
        connection=connection,
        results=nli_news_results,
    )

    print(
        f"Resultados procesados: {len(nli_news_results)}"
    )

except Exception:
    connection.rollback()
    raise

finally:
    connection.close()

Resultados procesados: 50
